### 作业 1：福大教务处情报分析任务（Pandas）

将上一轮作业的作业 1 爬取到的数据进行分析与可视化。

#### 要求

1. 请使用 **Jupyter Notebook** 的形式完成你的代码与分析报告，这能让你的分析过程一目了然。
2. **「通知人」都有谁？** 统计所有出现过的「通知人」，并计算他们各自发布的通知数量占总数的比例。
3. **附件下载次数与通知人的关系？** 分析附件的下载次数与通知人是否存在某种联系。比如，是不是某些特定部门发布的通知，附件下载量总是特别高？
4. **通知发布的高峰期？** 统计每天发布的通知数量，分析一下，通常在学期的哪个时间段，通知会变得特别密集？
5. **自由探索** 根据你对数据的好奇心，自行思考一个你感兴趣的问题，并进行数据分析。（例如：通知的标题长度和阅读量 / 下载量有关吗？标题中出现哪些关键词会更受关注？）
6. **图表可视化** 尝试使用 `matplotlib` 或其他可视化工具，将你在问题 **3** 和 **4** 中的分析结果以图表的形式呈现出来。

In [1]:
import pandas as pd
pd.options.mode.copy_on_write = True

announcements = pd.read_csv("announcements.csv")

_dtype = {
    # IDs / codes (preserve leading zeros)
    "id": pd.StringDtype(),
    "attachment_id": pd.StringDtype(),
    "attachment_file_code": pd.StringDtype(),
    "attachment_owner": pd.StringDtype(),
    # text
    "url": pd.StringDtype(),
    "title": pd.StringDtype(),
    "issuer": pd.StringDtype(),
    "body": pd.StringDtype(),
    "attachment_name": pd.StringDtype(),
    "attachment_url": pd.StringDtype(),
    # counts (nullable ints)
    "attachment_count": pd.Int64Dtype(),
    "attachment_download_times": pd.Int64Dtype(),
}
_parse_dates = ["pub_date"]

df = pd.read_csv("announcements.csv", dtype=_dtype, parse_dates=_parse_dates)  # pyright: ignore[reportArgumentType]

df["pub_date"] = pd.to_datetime(df["pub_date"], errors="coerce").dt.normalize()
df["pub_month"] = df["pub_date"].dt.month
df["pub_year"] = df["pub_date"].dt.year


In [14]:
out = df.groupby("issuer")["id"].nunique().sort_values(ascending=False).reset_index(name="count")
out["percentage"] = out["count"] / out["count"].sum()

out

,issuer,count,percentage
0,【教学运行】,170,0.340
1,【实践科】,106,0.212
2,【质量办】,69,0.138
3,【计划科】,46,0.092
4,【教研教改】,37,0.074
5,【综合科】,37,0.074
6,【教材中心】,27,0.054
7,【教学通知】,7,0.014
8,【电教中心】,1,0.002


可以看到，教学运行的公告数量最多，其次是实践科，两者综合超过一半

In [ ]:
attachment_data = df.dropna(subset=["attachment_id"])
download_times_data = attachment_data.groupby("issuer")["attachment_download_times"]

today = pd.Timestamp("2025-01-14").normalize()
attachment_data["pub_date"] = pd.to_datetime(attachment_data["pub_date"], errors="coerce").dt.normalize()
attachment_data["past_days"] = (today - attachment_data["pub_date"]).dt.days

attachment_data_elder = attachment_data[attachment_data["past_days"] >= 90].copy()
elder_download_times_data = attachment_data_elder.groupby("issuer")["attachment_download_times"]

stats = download_times_data.agg(
    total="sum",
    max="max",
    min="min",
    mean="mean", 
    median="median",
    std="std",
    p90=lambda x: x.quantile(0.9)  # pyright: ignore[reportUnknownMemberType, reportUnknownLambdaType]
)

elder_stats = elder_download_times_data.agg(
    total="sum",
    max="max",
    min="min",
    mean="mean", 
    median="median",
    std="std",
    p90=lambda x: x.quantile(0.9)  # pyright: ignore[reportUnknownMemberType, reportUnknownLambdaType]
)

# connect 2 dfs
combined_stats = elder_stats.merge(stats, on="issuer", how="outer", suffixes=("_elder", "_all"))  # pyright: ignore[reportAny]

combined_stats.sort_values(by="mean_elder", ascending=False)  # pyright: ignore[reportAny]


,total_elder,max_elder,min_elder,mean_elder,median_elder,std_elder,p90_elder,total_all,max_all,min_all,mean_all,median_all,std_all,p90_all
issuer,,,,,,,,,,,,,,
【计划科】,42850,5217,781,2520.588235,2372.0,1377.081563,4651.6,62111,5217,638,2070.366667,1718.0,1258.0937,3562.3
【实践科】,112279,9567,630,2339.145833,1313.0,2188.924607,4590.3,161687,9567,31,1684.239583,989.5,1872.230085,3934.0
【教研教改】,51960,4010,607,1443.333333,1103.5,820.874273,2534.0,61466,4010,47,1254.408163,971.0,825.273059,2209.8
【教学运行】,44718,1676,617,912.612245,854.0,232.807218,1194.8,104152,9112,35,820.094488,718.0,1015.089209,1230.8
【质量办】,21792,1593,557,871.68,784.0,298.343024,1392.8,33542,1593,85,632.867925,602.0,341.695098,965.4
【教材中心】,11632,1114,69,775.466667,821.0,285.28003,1062.2,19568,1114,15,337.37931,105.0,375.621115,955.9
【教学通知】,2522,828,518,630.5,588.0,140.205801,768.9,3683,828,218,526.142857,545.0,199.403635,709.8
【综合科】,1734,467,373,433.5,447.0,42.21769,464.0,3487,467,170,348.7,370.5,110.036408,458.9


In [ ]:

(df.groupby("pub_month", as_index=True)  # pyright: ignore[reportUnusedCallResult]
    .agg(count=("id", "nunique")))

,count
pub_month,
1,40
2,20
3,32
4,57
5,49
6,54
7,13
8,28
9,73


#### 我的问题

通知数量随年份变化是否存在变化？若存在，趋势如何？

In [2]:
(df.groupby("pub_year", as_index=True)  # pyright: ignore[reportUnusedCallResult]
    .agg(count=("id", "nunique")))

,count
pub_year,
2023,57
2024,230
2025,198
2026,15
